- https://verl.readthedocs.io/en/latest/advance/agent_loop.html
- https://github.com/volcengine/verl/blob/main/verl/experimental/agent_loop/agent_loop.py
    - AgentLoopManager, AgentLoopWorker, AgentLoopBase, AsyncLLMServerManager
    - 相信这些设计是 general 的，同样地适用到其他的 rl infra lib 中的；
    - inference 测试
        - https://github.com/volcengine/verl/blob/main/tests/experimental/agent_loop/test_basic_agent_loop.py
- 数据准备与训练示例
    - https://github.com/volcengine/verl/tree/main/examples/data_preprocess
        - multi-turn 相关的
    - https://github.com/volcengine/verl/tree/main/examples/sglang_multiturn
        - `actor_rollout_ref.rollout.mode=async`
        - https://verl.readthedocs.io/en/latest/start/agentic_rl.html

### agent_loop.py: AgentLoopManager

```python
def generate_sequences(self, prompts: DataProto) -> DataProto:
    """Split input batch and dispatch to agent loop workers.

    Args:
        prompts (DataProto): Input batch.

    Returns:
        DataProto: Output batch.
    """

    if self.config.actor_rollout_ref.rollout.free_cache_engine:
        self.wake_up()
    if self.reward_model_manager and self.config.reward_model.rollout.free_cache_engine:
        self.reward_model_manager.wake_up()

    chunkes = prompts.chunk(len(self.agent_loop_workers))
    outputs = ray.get(
        [
            worker.generate_sequences.remote(chunk)
            for worker, chunk in zip(self.agent_loop_workers, chunkes, strict=True)
        ]
    )
    output = DataProto.concat(outputs)
    if self.config.actor_rollout_ref.rollout.free_cache_engine:
        self.sleep()
    if self.reward_model_manager and self.config.reward_model.rollout.free_cache_engine:
        self.reward_model_manager.sleep()

    # calculate performance metrics
    metrics = [output.meta_info.pop("metrics") for output in outputs]  # List[List[Dict[str, str]]]
    timing = self._performance_metrics(metrics, output)

    output.meta_info = {"timing": timing, **outputs[0].meta_info}
    return output
```

### ray_trainer.py

- ray_trainer.py
    - init_workers
        - `self.config.actor_rollout_ref.rollout.mode == "async":`
        - 实例化 `self.async_rollout_manager = AgentLoopManager(`
    - fit
        - `gen_batch_output = self.async_rollout_manager.generate_sequences(gen_batch_output)`
    - _validate
        - `self.async_rollout_manager.generate_sequences(test_gen_batch_padded)`